## What to Vary

In [1]:
# language="english"
# language="multilingual"
# DeepPavlov/rubert-base-cased-sentence


# raw text or vw text


# default topics (whatever)
# specific number of topics


# KeyBERTInspired
# openchat

In [2]:
from topicnet.cooking_machine import Dataset

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

from umap import UMAP
from hdbscan import HDBSCAN

from hdbscan.flat import HDBSCAN_flat

from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora

from gensim.models.coherencemodel import CoherenceModel

import pandas as pd

In [3]:
import nltk
from nltk.corpus import stopwords
 
nltk.download('stopwords')

print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/alekseev_v/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/MKB10.csv',
)

dataset.get_possible_modalities()

{'@letter', '@ngram', '@text'}

In [7]:
MAIN_MODALITY = '@text'

In [8]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
«Бедная_симптомами»_шизофрения,«Бедная_симптомами»_шизофрения,«Бе́дная симпто́мами» шизофрени́я — подтип шиз...,«Бедная_симптомами»_шизофрения |@text бедный с...
"46,XX/46,XY","46,XX/46,XY","46,XX/46,XY (тетрагаметный химеризм) — это раз...","46,XX/46,XY |@text <person> химеризм разновидн..."
"Синдром_48,_XXXY","Синдром_48,_XXXY","Синдром 48, XXXY — это генетическое состояние,...","Синдром_48,_XXXY |@text синдром xxxy генетичес..."
"Синдром_48,_XXYY","Синдром_48,_XXYY","Синдром 48, XXYY — это аномалия хромосом, при ...","Синдром_48,_XXYY |@text синдром xxyy аномалия ..."
"Синдром_48,_XYYY","Синдром_48,_XYYY","Синдром 48, XYYY — чрезвычайно редкая анеуплои...","Синдром_48,_XYYY |@text синдром xyyy чрезвычаи..."


In [9]:
dataset._data.shape

(2036, 3)

In [10]:
dataset._data.dropna(axis=0, inplace=True)

In [11]:
dataset._data.shape

(2036, 3)

In [12]:
dataset._data['raw_text']

id
«Бедная_симптомами»_шизофрения                   «Бе́дная симпто́мами» шизофрени́я — подтип шиз...
46,XX/46,XY                                      46,XX/46,XY (тетрагаметный химеризм) — это раз...
Синдром_48,_XXXY                                 Синдром 48, XXXY — это генетическое состояние,...
Синдром_48,_XXYY                                 Синдром 48, XXYY — это аномалия хромосом, при ...
Синдром_48,_XYYY                                 Синдром 48, XYYY — чрезвычайно редкая анеуплои...
                                                                       ...                        
Синдром_SERKAL                                   Синдром SERKAL — аутосомно-рецессивное заболев...
VLDLR-ассоциированная_мозжечковая_гипоплазия     VLDLR-ассоциированная мозжечковая гипоплазия (...
X-связанная_эндотелиальная_дистрофия_роговицы    X-связанная, или X-сцепленная, эндотелиальная ...
X-связанный_ихтиоз                               X-связанный ихтиоз (X-сцепленный ихтиоз) — X-с...
XX-дисг

In [13]:
docs = list(dataset._data['raw_text'].values)

In [14]:
docs[:3]

['«Бе́дная симпто́мами» шизофрени́я\xa0— подтип шизотипического расстройства в российской версии МКБ-10[1] (ранее считавшийся «простым вариантом вялопротекающей шизофрении»[2][3] и «первичным дефект-психозом»[4][3]), проявляющийся преимущественно негативными симптомами (апатией, астеническим дефектом, суженным или уплощённым аффектом, социальной аутизацией, но без бреда и галлюцинаций).\nОсновные характеристики этого заболевания\xa0— нарастающий аутизм, снижение продуктивности деятельности, обеднение влечений, сужение диапазона эмоциональных реакций и явления астенического дефекта (пассивность, вялость, безынициативность)[1].\nДанное расстройство возникает чаще всего у личностей, характеризующихся замкнутостью и безынициативностью, лишённых эмоциональных привязанностей, которым с детства как бы не хватает «жизненной энергии»[3]. В латентном периоде заболевания медленно углубляется психическая дефицитарность, то есть снижается психическая активность, инициатива, возникает эмоциональная 

In [15]:
NUM_TOP_WORDS = 20

In [16]:
import torch
import transformers
import os

import json
import numpy as np

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [17]:
def get_phi(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    # ptw = np.array(mtw[1:, :])
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T
    vocabulary = topic_model.vectorizer_model.get_feature_names_out()

    assert pwt.shape[0] == len(vocabulary)

    # topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    # topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(pwt.shape[1] - 1)]

    phi = pd.DataFrame(
        index=vocabulary,
        columns=topic_names,
        data=pwt,
    )

    return phi


def get_top_words(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T

    # topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(pwt.shape[1] - 1)]
    topic_top_words = {
        n: topic_model.get_topic(t)
        for t, n in zip([-1] + list(range(NUM_TOPICS)), topic_names)
    }

    return topic_top_words


def get_dataset(topic_model, dataset, docs):
    cleaned_docs = topic_model._preprocess_text(docs)
    vectorizer = topic_model.vectorizer_model
    tokenizer = vectorizer.build_tokenizer()
    doc_tokens = [tokenizer(doc) for doc in cleaned_docs]
    doc_texts = [
        d + f' |{MAIN_MODALITY} ' + ' '.join(t)
        for d, t in zip(dataset._data.index, doc_tokens)
    ]
    data = [[d, t] for d, t in zip(dataset._data.index, doc_texts)]

    new_dataset = pd.DataFrame(
        columns=['id', 'vw_text'],
        data=data,
    )

    return new_dataset

In [18]:
NUM_TOPICS = 50
NUM_TOP_WORDS = 20
NUM_TRAINS = 20
STOP_WORDS = stopwords.words('russian')
LANGUAGE = 'multilingual'

In [19]:
! ls ../results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [21]:
SAVE_FOLDER = os.path.join('/data_mil/shared/CompressaAI/BERTopic', 'results50', 'mkb10')

In [22]:
! mkdir -p $SAVE_FOLDER

In [23]:
SAVE_FOLDER

'/data_mil/shared/CompressaAI/BERTopic/results50/mkb10'

In [24]:
for seed in range(NUM_TRAINS):
    print(seed)

    seed_save_folder = os.path.join(SAVE_FOLDER, str(seed))

    if os.path.isdir(seed_save_folder):
        contents = os.listdir(seed_save_folder)

        assert len(contents) == 3

        continue

    os.makedirs(seed_save_folder)

    keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)
    mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)
    
    representation_model = {
        "KeyBERT": keybert,
        "MMR": mmr,
    }

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    vectorizer_model = CountVectorizer(stop_words=STOP_WORDS)

    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,

        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        # hdbscan_model=hdbscan_model,            # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )
        
    topics, probs = topic_model.fit_transform(docs)
    orig_num_topics = len(set(topic_model.topics_))
    doc_embeddings = topic_model.umap_model.embedding_

    hdbscan_model = HDBSCAN_flat(doc_embeddings, n_clusters=NUM_TOPICS)
    
    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,
        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )

    topics, probs = topic_model.fit_transform(docs)
    
    new_num_topics = len(set(topic_model.topics_))
    
    # assert new_num_topics < orig_num_topics
    if new_num_topics >= orig_num_topics:
        print(f'No less topics: {new_num_topics} >= {orig_num_topics}.')

    # assert new_num_topics == NUM_TOPICS + 
    if new_num_topics != NUM_TOPICS + 1:
        print(f'WTF: failed to produce exact number of topics: {new_num_topics} != {NUM_TOPICS + 1}.')

        assert abs(new_num_topics - (NUM_TOPICS + 1)) <= 2

    assert topic_model.c_tf_idf_.shape[0] == new_num_topics
    
    phi = get_phi(topic_model)
    top_words = get_top_words(topic_model)
    new_dataset = get_dataset(topic_model, dataset, docs)
    
    phi.to_csv(f'{seed_save_folder}/phi.csv')
    
    with open(f'{seed_save_folder}/top_words.json', 'w') as f:
        f.write(
            json.dumps(
                top_words, indent=4, ensure_ascii=False
            )
        )
    
    new_dataset.to_csv(f'{seed_save_folder}/dataset.csv')

    del topic_model, phi, new_dataset

2024-03-30 10:15:35,599 - BERTopic - Embedding - Transforming documents to embeddings.


0


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

2024-03-30 10:15:46,400 - BERTopic - Embedding - Completed ✓
2024-03-30 10:15:46,401 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 10:15:58,826 - BERTopic - Dimensionality - Completed ✓
2024-03-30 10:15:58,827 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 10:15:59,114 - BERTopic - Cluster - Completed ✓
2024-03-30 10:15:59,122 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 10:16:06,910 - BERTopic - Representation - Completed ✓
2024-03-30 10:16:09,479 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

2024-03-30 10:32:27,028 - BERTopic - Embedding - Completed ✓
2024-03-30 10:32:27,029 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 10:32:33,603 - BERTopic - Dimensionality - Completed ✓
2024-03-30 10:32:33,604 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 10:32:33,850 - BERTopic - Cluster - Completed ✓
2024-03-30 10:32:33,853 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 10:32:41,162 - BERTopic - Representation - Completed ✓
2024-03-30 10:32:42,942 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

2024-03-30 10:32:49,580 - BERTopic - Embedding - Completed ✓
2024-03-30 10:32:49,581 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 10:32:56,123 - BERTopic - Dimensionality - Completed ✓
2024-03-30 10:32:56,124 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 10:32:56,742 - BERTopic - Cluster - Completed ✓
2024-03-30 10:32:56,745 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 10:33:04,724 - BERTopic - Representation - Completed ✓


No less topics: 51 >= 45.


In [25]:
! ls $SAVE_FOLDER

0  1  10  11  12  13  14  15  16  17  18  19  2  3  4  5  6  7	8  9


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [45]:
new_num_topics

52

In [31]:
SAVE_FOLDER

'/data_mil/shared/CompressaAI/BERTopic/results50/rtlwikiperson'

In [54]:
! ls $SAVE_FOLDER

0  1  10  11  2  3  4  5  6  7	8  9


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
! ls /data_mil/shared/CompressaAI/BERTopic/results50/rtlwikiperson/

0  1  10  11  2  3  4  5  6  7	8  9


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [48]:
! ls /data_mil/shared/CompressaAI/BERTopic/results50/rtlwikiperson/11

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
